In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import roc_auc_score, classification_report, brier_score_loss, log_loss
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df_geral = pd.read_csv('status_individual_emea.csv')

In [2]:
df_geral = pd.read_csv('status_individual_emea.csv')

# Remove valores duplicados
df_geral = df_geral.drop(
    columns=[
        'Time1',
        'Mapas_T1',
        'Score_T1',
        'Time2',
        'Mapas_T2',
        'Score_T2'
    ]
)
df_geral = df_geral.drop_duplicates()
df_geral = df_geral.reset_index(drop=True)

#Adicionar role(função) para cada jogador
agentes_valorant = {"duelistas": ["Jett", "Phoenix", "Reyna", "Raze", "Yoru", "Neon", "Iso", "Waylay"],
                    "controladores": ["Brimstone", "Viper", "Omen", "Astra", "Harbor", "Clove"],
                    "iniciadores": ["Sova", "Breach", "Skye", "Kayo", "Fade", "Gekko", "Tejo"],
                    "sentinelas": ["Sage", "Cypher", "Killjoy", "Chamber", "Deadlock", "Vyse", "Veto"]}

for i in range(len(df_geral)):
    agente_jogador = df_geral.loc[i, 'Agente']
    for role, agentes in agentes_valorant.items():
        if agente_jogador in agentes:
            df_geral.loc[i, 'Role'] = role

# Adicionea as win_rate em cada jogador
df_geral["win_rate_jogador"] = (
    df_geral.groupby("Jogador")["target"].transform("mean")
)

df_geral["win_rate_agente"] = (
    df_geral.groupby("Agente")["target"].transform("mean")
)

df_geral["win_rate_mapa"] = (
    df_geral.groupby("Mapa")["target"].transform("mean")
)

df_geral["win_rate_jogador_agente"] = (
    df_geral.groupby(["Jogador", "Agente"])["target"].transform("mean")
)

df_geral["win_rate_jogador_mapa"] = (
    df_geral.groupby(["Jogador", "Mapa"])["target"].transform("mean")
)


#Calcula o Impacto do jogador em seu time de acrodo com usa função
colunas_stats = ['R', 'ACS', 'K', 'D', 'A', '+/-', 'KAST', 'ADR', 'HS%', 'FK', 'FD', '+/-_FK_FD']

peso_role = {
    'duelistas':     {'R': 1.20, 'ACS': 1.20, 'K': 1.15, 'D': 1.00, 'A': 1.05, '+/-': 1.00, 'KAST': 1.10, 'ADR': 1.15, 'HS%': 1.00, 'FK': 1.15, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'iniciadores':   {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'controladores': {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.25, 'HS%': 1.00, 'FK': 1.00, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'sentinelas':    {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.15, '+/-': 1.00, 'KAST': 1.30, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
}

df_geral['Impacto'] = 0.0
df_geral['Impacto_Time_%'] = 0.0

for i in range(0, len(df_geral), 5):
    grupo = df_geral.iloc[i:i+5]
    impactos = []
    for _, row in grupo.iterrows():
        pesos = peso_role[row['Role']]
        impacto = sum(row[col] * pesos[col] for col in colunas_stats)
        impactos.append(impacto)

    total = sum(impactos)
    for j, idx in enumerate(grupo.index):
        df_geral.loc[idx, 'Impacto']       = round(impactos[j], 3)
        df_geral.loc[idx, 'Impacto_Time_%'] = round(impactos[j] / total * 100, 3)

df_geral

,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,target,Role,win_rate_jogador,win_rate_agente,win_rate_mapa,win_rate_jogador_agente,win_rate_jogador_mapa,Impacto,Impacto_Time_%
0,0,Ascent,sheydos,MKOI,Killjoy,2.02,322.0,27,9,2,18,80.0,199.0,31.0,2.0,1.0,1.0,1,sentinelas,0.336842,0.469048,0.476471,0.423077,0.375000,777.323,29.461
1,0,Ascent,trexx,MKOI,Kayo,1.15,189.0,12,13,12,-1,70.0,124.0,12.0,3.0,0.0,3.0,1,iniciadores,0.532110,0.471649,0.476471,0.380952,0.666667,493.822,18.716
2,0,Ascent,starxo,MKOI,Fade,1.07,196.0,14,13,9,1,75.0,131.0,30.0,0.0,1.0,-1.0,1,iniciadores,0.420000,0.515658,0.476471,0.562500,0.400000,528.181,20.019
3,0,Ascent,koldamenta,MKOI,Omen,0.92,120.0,7,12,10,-5,60.0,86.0,17.0,0.0,0.0,0.0,1,controladores,0.343750,0.487709,0.476471,0.200000,0.428571,353.258,13.389
4,0,Ascent,Wolfen,MKOI,Jett,0.59,185.0,12,16,3,-4,55.0,112.0,32.0,6.0,7.0,-1.0,1,duelistas,0.343750,0.471239,0.476471,0.411765,0.428571,485.858,18.415
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8325,5,Pearl,Boo,TH,Astra,1.00,240.0,17,16,4,1,65.0,144.0,16.0,1.0,0.0,1.0,0,controladores,0.578035,0.504559,0.479592,0.388889,0.545455,572.900,24.386
8326,5,Pearl,Wo0t,TH,Jett,0.99,196.0,15,14,1,1,70.0,112.0,27.0,2.0,3.0,-1.0,0,duelistas,0.621212,0.471239,0.479592,0.555556,0.571429,506.788,21.572
8327,5,Pearl,RieNs,TH,Fade,0.78,173.0,11,15,6,-4,65.0,119.0,50.0,1.0,3.0,-2.0,0,iniciadores,0.617450,0.515658,0.479592,0.592593,0.571429,488.947,20.812
8328,5,Pearl,benjyfishy,TH,Cypher,0.66,135.0,10,16,4,-6,50.0,99.0,36.0,0.0,3.0,-3.0,0,sentinelas,0.611842,0.520635,0.479592,0.625000,0.625000,387.909,16.512


In [3]:
df_ml = df_geral.copy()
le_jogador = LabelEncoder()
df_ml['Jogador_enc'] = le_jogador.fit_transform(df_ml['Jogador'])


# Remove as colunas OHE antigas antes de recriar
cols_ohe_existentes = [c for c in df_ml.columns if c.startswith('Mapa_') or c.startswith('Agente_')]
df_ml = df_ml.drop(columns=cols_ohe_existentes)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

ohe_array = ohe.fit_transform(df_ml[['Mapa', 'Agente']])
ohe_cols  = ohe.get_feature_names_out(['Mapa', 'Agente'])
df_ohe    = pd.DataFrame(ohe_array, columns=ohe_cols, index=df_ml.index)
df_ml     = pd.concat([df_ml, df_ohe], axis=1)

# Atualiza a lista de features
features_numericas = [
    'R', 'ACS', 'K', 'D', 'A', '+/-', 'KAST', 'ADR', 'HS%', 'FK', 'FD', '+/-_FK_FD',
    'Impacto', 'Impacto_Time_%',
    'win_rate_jogador', 'win_rate_agente', 'win_rate_mapa',
    'win_rate_jogador_agente', 'win_rate_jogador_mapa'
]
features1 = features_numericas + ['Jogador_enc'] + list(ohe_cols)

K_SUAV = 10
global_mean = df_geral['target'].mean()

def win_rate_suavizada(df_f, col, global_mean, k=K_SUAV):
    n = len(df_f)
    if n == 0:
        return global_mean
    return (df_f[col].sum() + k * global_mean) / (n + k)

for idx, row in df_ml.iterrows():
    jogador = row['Jogador']
    agente  = row['Agente']
    mapa    = row['Mapa']

    f_j  = df_geral[df_geral['Jogador'] == jogador]
    f_a  = df_geral[df_geral['Agente']  == agente]
    f_m  = df_geral[df_geral['Mapa']    == mapa]
    f_ja = f_j[f_j['Agente'] == agente]
    f_jm = f_j[f_j['Mapa']   == mapa]

    df_ml.at[idx, 'win_rate_jogador']        = win_rate_suavizada(f_j,  'target', global_mean)
    df_ml.at[idx, 'win_rate_agente']         = win_rate_suavizada(f_a,  'target', global_mean)
    df_ml.at[idx, 'win_rate_mapa']           = win_rate_suavizada(f_m,  'target', global_mean)
    df_ml.at[idx, 'win_rate_jogador_agente'] = win_rate_suavizada(f_ja, 'target', global_mean)
    df_ml.at[idx, 'win_rate_jogador_mapa']   = win_rate_suavizada(f_jm, 'target', global_mean)

# Treinar Modelo
X1 = df_ml[features1]
y1 = df_ml['target']

X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1
)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=10,
    min_samples_leaf=5, class_weight='balanced',
    random_state=42, n_jobs=-1
)
modelo1 = CalibratedClassifierCV(rf, method='isotonic', cv=5)
modelo1.fit(X_train1, y_train1)

# Prever Modelo
y_pred1  = modelo1.predict(X_test1)
y_proba1 = modelo1.predict_proba(X_test1)[:, 1]

print(f"ROC-AUC : {roc_auc_score(y_test1, y_proba1):.4f}")
print(f"Log Loss: {log_loss(y_test1, y_proba1):.4f}")
print(f"Brier   : {brier_score_loss(y_test1, y_proba1):.4f}")
print(classification_report(y_test1, y_pred1))

cv_scores = cross_val_score(
    modelo1, X1, y1,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc'
)
print(f"AUC CV  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

ROC-AUC : 0.9347
Log Loss: 0.3225
Brier   : 0.1042
              precision    recall  f1-score   support

           0       0.86      0.84      0.85       852
           1       0.84      0.85      0.84       814

    accuracy                           0.85      1666
   macro avg       0.85      0.85      0.85      1666
weighted avg       0.85      0.85      0.85      1666

AUC CV  : 0.9283 ± 0.0028


# Previsoes de outros Jogos

In [4]:
# ── Constantes ─────────────────────────────────────────────────────────────────
colunas_stats = ['R', 'ACS', 'K', 'D', 'A', '+/-', 'KAST', 'ADR', 'HS%', 'FK', 'FD', '+/-_FK_FD']
peso_role = {
    'duelistas':     {'R': 1.20, 'ACS': 1.20, 'K': 1.15, 'D': 1.00, 'A': 1.05, '+/-': 1.00, 'KAST': 1.10, 'ADR': 1.15, 'HS%': 1.00, 'FK': 1.15, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'iniciadores':   {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'controladores': {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.25, 'HS%': 1.00, 'FK': 1.00, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'sentinelas':    {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.15, '+/-': 1.00, 'KAST': 1.30, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
}

# ── Funções ────────────────────────────────────────────────────────────────────
def calcular_impacto(df_time: pd.DataFrame) -> pd.DataFrame:
    df       = df_time.copy()
    impactos = [
        sum(row[col] * peso_role[row['Role']][col] for col in colunas_stats)
        for _, row in df.iterrows()
    ]
    total                = sum(impactos)
    df['Impacto']        = [round(v, 3) for v in impactos]
    df['Impacto_Time_%'] = [round(v / total * 100, 3) for v in impactos]
    return df

def obter_historico(jogador, agente, mapa, df_ml):
    tentativas = [
        df_ml[(df_ml['Jogador'] == jogador) & (df_ml['Agente'] == agente) & (df_ml['Mapa'] == mapa)],
        df_ml[(df_ml['Jogador'] == jogador) & (df_ml['Agente'] == agente)],
        df_ml[(df_ml['Jogador'] == jogador) & (df_ml['Mapa']   == mapa)],
        df_ml[(df_ml['Jogador'] == jogador)],
    ]

    for df_f in tentativas:
        if not df_f.empty:
            row = df_f.iloc[0]
            return {
                'win_rate_jogador':        row['win_rate_jogador'],
                'win_rate_agente':         row['win_rate_agente'],
                'win_rate_mapa':           row['win_rate_mapa'],
                'win_rate_jogador_agente': row['win_rate_jogador_agente'],
                'win_rate_jogador_mapa':   row['win_rate_jogador_mapa'],
                'Jogador_enc':             row['Jogador_enc'],
                'Role':                    row['Role'],
            }

    global_mean = df_ml['target'].mean()
    return {
        'win_rate_jogador':        global_mean,
        'win_rate_agente':         global_mean,
        'win_rate_mapa':           global_mean,
        'win_rate_jogador_agente': global_mean,
        'win_rate_jogador_mapa':   global_mean,
        'Jogador_enc':             -1,
        'Role':                    None,
    }

def prever_jogador(jogador, agente, mapa, stats_row, df_ml, modelo, ohe, ohe_cols, features):
    hist = obter_historico(jogador, agente, mapa, df_ml)

    features_row = {
        'R':              stats_row['R'],
        'ACS':            stats_row['ACS'],
        'K':              stats_row['K'],
        'D':              stats_row['D'],
        'A':              stats_row['A'],
        '+/-':            stats_row['+/-'],
        'KAST':           stats_row['KAST'],
        'ADR':            stats_row['ADR'],
        'HS%':            stats_row['HS%'],
        'FK':             stats_row['FK'],
        'FD':             stats_row['FD'],
        '+/-_FK_FD':      stats_row['+/-_FK_FD'],
        'Impacto':        stats_row['Impacto'],
        'Impacto_Time_%': stats_row['Impacto_Time_%'],
        **{k: v for k, v in hist.items() if k != 'Role'},
    }

    ohe_input = ohe.transform([[mapa, agente]])
    features_row.update(dict(zip(ohe_cols, ohe_input[0])))

    X_in = pd.DataFrame([features_row])[features].fillna(0)
    prob = float(np.clip(modelo.predict_proba(X_in)[0][1], 0.05, 0.95))
    return prob

# ── Time ───────────────────────────────────────────────────────────────────────
MAPA = 'Haven'

time = pd.DataFrame([
    {'Jogador': 'Chronicle', 'Agente': 'Chamber', 'Role': 'sentinelas',    'R': 1.08, 'ACS': 211, 'K': 48, 'D': 43, 'A': 17, '+/-': 5,  'KAST': 72, 'ADR': 149, 'HS%': 23, 'FK': 8, 'FD': 9,  '+/-_FK_FD': -1},
    {'Jogador': 'PROFEK',    'Agente': 'Astra',   'Role': 'controladores', 'R': 1.05, 'ACS': 177, 'K': 42, 'D': 40, 'A': 29, '+/-': 2,  'KAST': 73, 'ADR': 117, 'HS%': 19, 'FK': 3, 'FD': 2,  '+/-_FK_FD': 1},
    {'Jogador': 'Sayonara',  'Agente': 'Sova',    'Role': 'iniciadores',   'R': 1.04, 'ACS': 218, 'K': 46, 'D': 46, 'A': 20, '+/-': 0,  'KAST': 73, 'ADR': 143, 'HS%': 21, 'FK': 7, 'FD': 10, '+/-_FK_FD': -3},
    {'Jogador': 'Derke',     'Agente': 'Neon',    'Role': 'duelistas',     'R': 0.95, 'ACS': 204, 'K': 45, 'D': 49, 'A': 13, '+/-': -4, 'KAST': 66, 'ADR': 134, 'HS%': 28, 'FK': 5, 'FD': 6,  '+/-_FK_FD': -1},
    {'Jogador': 'Jamppi',    'Agente': 'KAY/O',   'Role': 'iniciadores',   'R': 0.93, 'ACS': 206, 'K': 46, 'D': 49, 'A': 21, '+/-': -3, 'KAST': 78, 'ADR': 130, 'HS%': 21, 'FK': 5, 'FD': 9,  '+/-_FK_FD': -4},
])
time = calcular_impacto(time)

# ── Predição ───────────────────────────────────────────────────────────────────
resultados = []
for _, row in time.iterrows():
    prob = prever_jogador(
        jogador=row['Jogador'], agente=row['Agente'], mapa=MAPA,
        stats_row=row, df_ml=df_ml, modelo=modelo1,
        ohe=ohe, ohe_cols=ohe_cols, features=features1
    )
    jogos_hist = len(df_ml[(df_ml['Jogador'] == row['Jogador']) & (df_ml['Agente'] == row['Agente'])])
    resultados.append({
        'Jogador':          row['Jogador'],
        'Agente':           row['Agente'],
        'Role':             row['Role'],
        'Impacto_Time_%':   f"{row['Impacto_Time_%']:.1f}%",
        'Jogos_historicos': jogos_hist,
        'Prob_Vitoria':     f"{prob*100:.1f}%",
    })

df_resultado = pd.DataFrame(resultados).sort_values('Prob_Vitoria', ascending=False)
df_resultado

/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with f

,Jogador,Agente,Role,Impacto_Time_%,Jogos_historicos,Prob_Vitoria
1,PROFEK,Astra,controladores,18.3%,23,82.3%
0,Chronicle,Chamber,sentinelas,21.0%,7,77.4%
2,Sayonara,Sova,iniciadores,20.8%,8,77.3%
3,Derke,Neon,duelistas,19.9%,22,60.1%
4,Jamppi,KAY/O,iniciadores,20.0%,0,30.9%


# Previsao Jogador + Mapa + Agente

In [16]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import numpy as np


df_geral = pd.read_csv('status_individual_emea.csv')
df_geral = df_geral.drop(
    columns=[
        'Time1',
        'Mapas_T1',
        'Score_T1',
        'Time2',
        'Mapas_T2',
        'Score_T2'
    ]
)
df_geral = df_geral.drop_duplicates()
df_geral = df_geral.reset_index(drop=True)
df = df_geral.copy()

In [17]:
def smooth_target_encode(df, col, keys, global_mean, k=10):
    df[col] = np.nan

    for train_idx, val_idx in kf.split(df):
        treino = df.iloc[train_idx]

        stats = (
            treino.groupby(keys)["target"]
            .agg(["mean", "count"])
            .rename(columns={"mean": "grp_mean", "count": "grp_n"})
            .reset_index()
        )

        stats[col] = (
            (stats["grp_n"] * stats["grp_mean"] + k * global_mean)
            / (stats["grp_n"] + k)
        )
        stats = stats.drop(columns=["grp_mean", "grp_n"])

        df_val = df.iloc[val_idx][keys].copy().merge(stats, on=keys, how="left")
        df.loc[df.index[val_idx], col] = df_val[col].values

    df[col] = df[col].fillna(global_mean)


kf = KFold(n_splits=5, shuffle=True, random_state=42)
global_mean = df2["target"].mean()

grupos = {
    "win_rate_jogador":          ["Jogador"],
    "win_rate_agente":           ["Agente"],
    "win_rate_mapa":             ["Mapa"],
    "win_rate_jogador_agente":   ["Jogador", "Agente"],
    "win_rate_jogador_mapa":     ["Jogador", "Mapa"],
}

for col, keys in grupos.items():
    smooth_target_encode(df, col, keys, global_mean, k=10)

# Treinar Modelo
features2 = [
    "Jogador", "Mapa", "Agente",
    "win_rate_jogador", "win_rate_agente", "win_rate_mapa",
    "win_rate_jogador_agente", "win_rate_jogador_mapa",
]
 
X2 = df[features2].copy()
y2 = df["target"]


X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

encoder = TargetEncoder(cols=["Jogador", "Mapa", "Agente"], smoothing=10)
X_train2 = encoder.fit_transform(X_train2, y_train2)
X_test2  = encoder.transform(X_test2)

modelo2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
modelo2.fit(X_train2, y_train2)
global_mean_modelo2 = global_mean 

# Prever Modelo
y_prob2 = modelo2.predict_proba(X_test2)[:, 1]
y_pred2 = modelo2.predict(X_test2)
 
print(f"AUC-ROC : {roc_auc_score(y_test2, y_prob2):.3f}")
print(classification_report(y_test2, y_pred2))

AUC-ROC : 0.599
              precision    recall  f1-score   support

           0       0.59      0.52      0.55       852
           1       0.55      0.62      0.58       814

    accuracy                           0.57      1666
   macro avg       0.57      0.57      0.57      1666
weighted avg       0.57      0.57      0.57      1666



In [19]:
def prever_vitoria(jogador, mapa, agente, k=0):
    def get_wr_smooth(mask, col):
        vals = df.loc[mask, "target"] 
        n = len(vals)
        if n == 0:
            return global_mean
        return (n * vals.mean() + k * global_mean) / (n + k)

    wr_jogador       = get_wr_smooth(df.Jogador == jogador, "win_rate_jogador")
    wr_agente        = get_wr_smooth(df.Agente  == agente,  "win_rate_agente")
    wr_mapa          = get_wr_smooth(df.Mapa    == mapa,    "win_rate_mapa")
    wr_jog_agente    = get_wr_smooth((df.Jogador == jogador) & (df.Agente == agente), "win_rate_jogador_agente")
    wr_jog_mapa      = get_wr_smooth((df.Jogador == jogador) & (df.Mapa   == mapa),   "win_rate_jogador_mapa")

    entrada = pd.DataFrame([{
        "Jogador":               jogador,
        "Mapa":                  mapa,
        "Agente":                agente,
        "win_rate_jogador":      wr_jogador,
        "win_rate_agente":       wr_agente,
        "win_rate_mapa":         wr_mapa,
        "win_rate_jogador_agente": wr_jog_agente,
        "win_rate_jogador_mapa":   wr_jog_mapa,
    }])

    entrada_enc = encoder.transform(entrada).astype(float)
    prob = modelo2.predict_proba(entrada_enc)[0][1]

    print(f"Jogador : {jogador}")
    print(f"Mapa    : {mapa}")
    print(f"Agente  : {agente}")
    print(f"─────────────────────")
    print(f"Win prob: {prob:.1%}")
    return prob

In [20]:
print(prever_vitoria("Chronicle", "Pearl", "Chamber"))

Jogador : Chronicle
Mapa    : Pearl
Agente  : Chamber
─────────────────────
Win prob: 55.2%
0.5523314945534253


# Previsao Time (5 jogadores + 5 Agentes + 1 Mapa)

In [22]:
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np

df_geral = pd.read_csv('status_individual_emea.csv')


df_geral = df_geral.drop(
    columns=[
        'Time1',
        'Mapas_T1',
        'Score_T1',
        'Time2',
        'Mapas_T2',
        'Score_T2'
    ]
)
df_geral = df_geral.drop_duplicates()
df_geral = df_geral.reset_index(drop=True)

#Adicionar role(função) para cada jogador
agentes_valorant = {"duelistas": ["Jett", "Phoenix", "Reyna", "Raze", "Yoru", "Neon", "Iso", "Waylay"],
                    "controladores": ["Brimstone", "Viper", "Omen", "Astra", "Harbor", "Clove"],
                    "iniciadores": ["Sova", "Breach", "Skye", "Kayo", "Fade", "Gekko", "Tejo"],
                    "sentinelas": ["Sage", "Cypher", "Killjoy", "Chamber", "Deadlock", "Vyse", "Veto"]}

for i in range(len(df_geral)):
    agente_jogador = df_geral.loc[i, 'Agente']
    for role, agentes in agentes_valorant.items():
        if agente_jogador in agentes:
            df_geral.loc[i, 'Role'] = role

# Adicionea as win_rate em cada jogador
df_geral["win_rate_jogador"] = (
    df_geral.groupby("Jogador")["target"].transform("mean")
)

df_geral["win_rate_agente"] = (
    df_geral.groupby("Agente")["target"].transform("mean")
)

df_geral["win_rate_mapa"] = (
    df_geral.groupby("Mapa")["target"].transform("mean")
)

df_geral["win_rate_jogador_agente"] = (
    df_geral.groupby(["Jogador", "Agente"])["target"].transform("mean")
)

df_geral["win_rate_jogador_mapa"] = (
    df_geral.groupby(["Jogador", "Mapa"])["target"].transform("mean")
)


#Calcula o Impacto do jogador em seu time de acrodo com usa função
colunas_stats = ['R', 'ACS', 'K', 'D', 'A', '+/-', 'KAST', 'ADR', 'HS%', 'FK', 'FD', '+/-_FK_FD']

peso_role = {
    'duelistas':     {'R': 1.20, 'ACS': 1.20, 'K': 1.15, 'D': 1.00, 'A': 1.05, '+/-': 1.00, 'KAST': 1.10, 'ADR': 1.15, 'HS%': 1.00, 'FK': 1.15, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'iniciadores':   {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'controladores': {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.25, 'HS%': 1.00, 'FK': 1.00, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'sentinelas':    {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.15, '+/-': 1.00, 'KAST': 1.30, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
}

df_geral['Impacto'] = 0.0
df_geral['Impacto_Time_%'] = 0.0

for i in range(0, len(df_geral), 5):
    grupo = df_geral.iloc[i:i+5]
    impactos = []
    for _, row in grupo.iterrows():
        pesos = peso_role[row['Role']]
        impacto = sum(row[col] * pesos[col] for col in colunas_stats)
        impactos.append(impacto)

    total = sum(impactos)
    for j, idx in enumerate(grupo.index):
        df_geral.loc[idx, 'Impacto']       = round(impactos[j], 3)
        df_geral.loc[idx, 'Impacto_Time_%'] = round(impactos[j] / total * 100, 3)

df = df_geral.copy()
df = df.reset_index(drop=True)

df['id_time'] = df.index // 5
df

linhas = []

for _, grupo in df.groupby('id_time'):

    linha = {
        'Mapa': grupo['Mapa'].iloc[0],
        'target': grupo['target'].iloc[0]
    }

    for i, (_, row) in enumerate(grupo.iterrows(), start=1):

        linha[f'Jogador{i}'] = row['Jogador']
        linha[f'Agente{i}'] = row['Agente']
        linha[f'Impacto{i}'] = row['Impacto']
        linha[f'WR_Jogador{i}'] = row['win_rate_jogador']
        linha[f'WR_Agente{i}'] = row['win_rate_agente']

    linhas.append(linha)

df = pd.DataFrame(linhas)

df

# Treianr Modelo

features3 = [
    'Mapa',
    'Jogador1','Jogador2','Jogador3','Jogador4','Jogador5',
    'Agente1','Agente2','Agente3','Agente4','Agente5'
]

encoders = {}

for col in features3:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

X3 = df.drop('target', axis=1)
y3 = df['target']

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3,
    y3,
    test_size=0.2,
    random_state=42,
    stratify=y3
)

modelo3 = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    random_state=42
)
modelo3.fit(X_train3, y_train3)

#Prever Modelo
y_prob3 = modelo3.predict_proba(X_test3)[:, 1]
y_pred3 = modelo3.predict(X_test3)
 
print(f"AUC-ROC : {roc_auc_score(y_test3, y_prob3):.3f}")
print(classification_report(y_test3, y_pred3))

AUC-ROC : 0.899
              precision    recall  f1-score   support

           0       0.83      0.78      0.81       171
           1       0.79      0.83      0.81       163

    accuracy                           0.81       334
   macro avg       0.81      0.81      0.81       334
weighted avg       0.81      0.81      0.81       334



In [23]:
def buscar_stats(jogador, agente, df_ml):

    df_jog = df_ml[df_ml['Jogador'] == jogador]

    wr_jogador = df_jog['win_rate_jogador'].mean()

    df_ag = df_ml[df_ml['Agente'] == agente]

    wr_agente = df_ag['win_rate_agente'].mean()

    df_imp = df_ml[
        (df_ml['Jogador'] == jogador) &
        (df_ml['Agente'] == agente)
    ]

    if len(df_imp) > 0:
        impacto = df_imp['Impacto'].mean()
    else:
        impacto = df_jog['Impacto'].mean()

    return impacto, wr_jogador, wr_agente

#Função de Previsão
def prever_time(
    mapa,
    jogadores,
    agentes,
    model,
    encoders,
    df_ml
):

    linha = {'Mapa': mapa}

    for i in range(5):

        impacto, wr_jogador, wr_agente = buscar_stats(
            jogadores[i],
            agentes[i],
            df_ml
        )

        linha[f'Jogador{i+1}'] = jogadores[i]
        linha[f'Agente{i+1}'] = agentes[i]

        linha[f'Impacto{i+1}'] = impacto
        linha[f'WR_Jogador{i+1}'] = wr_jogador
        linha[f'WR_Agente{i+1}'] = wr_agente

    novo_time = pd.DataFrame([linha])

    for col in features3:
        novo_time[col] = encoders[col].transform(
            novo_time[col].astype(str)
        )

    prob = model.predict_proba(novo_time)

    print(f"Chance de vitória: {prob[0][1]*100:.2f}%")

    return prob[0][1]

In [24]:
df = df_geral.copy()

prever_time(
    mapa='Haven',
    jogadores=[
        'Chronicle',
        'PROFEK',
        'Sayonara',
        'Derke',
        'Jamppi'
    ],
    agentes=[
        'Chamber',
        'Astra',
        'Sova',
        'Neon',
        'Kayo'
    ],
    model=modelo3,
    encoders=encoders,
    df_ml=df
)

Chance de vitória: 68.41%


np.float64(0.6840537325456896)

# Junção dos 3 modelos
modelo1 = 60

modelo2 = 8

modelo3 = 11

In [51]:
# ══════════════════════════════════════════════════════════════════════════
# ENSEMBLE — Junção dos 3 modelos
# Modelo 1: stats individuais + OHE  (60 features)
# Modelo 2: win rates + TargetEncoder (8 features)
# Modelo 3: time agregado + LabelEncoder (11 features)
# ══════════════════════════════════════════════════════════════════════════

def _prever_vitoria_silencioso(jogador, mapa, agente, df, encoder, modelo2, global_mean, k=0):
    """
    Versão silenciosa de prever_vitoria (sem prints) para uso interno no ensemble.
    """
    def get_wr_smooth(mask):
        vals = df.loc[mask, "target"]
        n = len(vals)
        if n == 0:
            return global_mean
        return (n * vals.mean() + k * global_mean) / (n + k)

    entrada = pd.DataFrame([{
        "Jogador":                 jogador,
        "Mapa":                    mapa,
        "Agente":                  agente,
        "win_rate_jogador":        get_wr_smooth(df.Jogador == jogador),
        "win_rate_agente":         get_wr_smooth(df.Agente  == agente),
        "win_rate_mapa":           get_wr_smooth(df.Mapa    == mapa),
        "win_rate_jogador_agente": get_wr_smooth((df.Jogador == jogador) & (df.Agente == agente)),
        "win_rate_jogador_mapa":   get_wr_smooth((df.Jogador == jogador) & (df.Mapa   == mapa)),
    }])

    entrada_enc = encoder.transform(entrada).astype(float)
    return float(modelo2.predict_proba(entrada_enc)[0][1])


def _prever_time_silencioso(mapa, jogadores, agentes, modelo3, encoders, features3, df_ml):
    """
    Versão corrigida de prever_time: garante que predict_proba recebe
    exatamente as colunas que o modelo3 foi treinado (features3 + numéricas).
    """
    linha = {'Mapa': mapa}

    for i in range(5):
        impacto, wr_jogador, wr_agente = buscar_stats(jogadores[i], agentes[i], df_ml)
        linha[f'Jogador{i+1}']    = jogadores[i]
        linha[f'Agente{i+1}']     = agentes[i]
        linha[f'Impacto{i+1}']    = impacto
        linha[f'WR_Jogador{i+1}'] = wr_jogador
        linha[f'WR_Agente{i+1}']  = wr_agente

    novo_time = pd.DataFrame([linha])

    # Codifica apenas as colunas categóricas (features3)
    for col in features3:
        try:
            novo_time[col] = encoders[col].transform(novo_time[col].astype(str))
        except ValueError:
            # Jogador/Agente/Mapa desconhecido → usa -1
            novo_time[col] = -1

    # ✅ Garante que o modelo recebe exatamente as colunas do treino (X3)
    colunas_modelo3 = modelo3.feature_names_in_
    for col in colunas_modelo3:
        if col not in novo_time.columns:
            novo_time[col] = 0.0

    return float(modelo3.predict_proba(novo_time[colunas_modelo3])[0][1])


def ensemble_prever_time(
    mapa,
    jogadores,   # lista com 5 nomes
    agentes,     # lista com 5 agentes
    roles,       # lista com 5 roles
    stats_time,  # lista com 5 dicts de stats
    # artefatos modelo 1
    modelo1, ohe, ohe_cols, features1, df_ml,
    # artefatos modelo 2
    modelo2, encoder, df_m2, global_mean_m2,
    # artefatos modelo 3
    modelo3, encoders, features3,
    # pesos (não precisam somar 1, são normalizados automaticamente)
    pesos=(0.4, 0.3, 0.3),
    k=0,
):
    w1, w2, w3 = pesos
    w_total    = w1 + w2 + w3

    # ── Monta DataFrame do time e calcula impacto ──────────────────
    df_time = pd.DataFrame([
        {'Jogador': jogadores[i], 'Agente': agentes[i], 'Role': roles[i], **stats_time[i]}
        for i in range(5)
    ])
    df_time = calcular_impacto(df_time)

    # ══════════════════════════════════════════════════════════════
    # MODELO 1 — Stats individuais por jogador
    # ══════════════════════════════════════════════════════════════a
    probs_m1 = []
    for _, row in df_time.iterrows():
        p = prever_jogador(
            jogador=row['Jogador'], agente=row['Agente'], mapa=mapa,
            stats_row=row, df_ml=df_ml, modelo=modelo1,
            ohe=ohe, ohe_cols=ohe_cols, features=features1
        )
        probs_m1.append(p)
    prob_m1 = float(np.mean(probs_m1))

    # ══════════════════════════════════════════════════════════════
    # MODELO 2 — Win rates jogador/agente/mapa
    # ══════════════════════════════════════════════════════════════
    probs_m2 = [
        _prever_vitoria_silencioso(
            jogadores[i], mapa, agentes[i],
            df_m2, encoder, modelo2, global_mean_m2, k=k
        )
        for i in range(5)
    ]
    prob_m2 = float(np.mean(probs_m2))

    # ══════════════════════════════════════════════════════════════
    # MODELO 3 — Time completo agregado
    # ══════════════════════════════════════════════════════════════
    prob_m3 = _prever_time_silencioso(
        mapa, jogadores, agentes, modelo3, encoders, features3, df_ml
    )

    # ══════════════════════════════════════════════════════════════
    # ENSEMBLE — Média ponderada normalizada
    # ══════════════════════════════════════════════════════════════
    prob_final = (w1 * prob_m1 + w2 * prob_m2 + w3 * prob_m3) / w_total

    # ── Relatório ─────────────────────────────────────────────────
    print(f"\n{'═'*50}")
    print(f"  ENSEMBLE VALORANT — {mapa}")
    print(f"{'═'*50}")
    print(f"  Modelo 1  stats individuais : {prob_m1:.1%}   peso={w1}")
    print(f"  Modelo 2  win rates         : {prob_m2:.1%}   peso={w2}")
    print(f"  Modelo 3  time agregado     : {prob_m3:.1%}   peso={w3}")
    print(f"{'─'*50}")
    print(f"  ► PROB. FINAL ENSEMBLE      : {prob_final:.1%}")
    print(f"{'═'*50}")
    print(f"\n  Detalhes por jogador (M1 | M2):")
    for i in range(5):
        print(f"    {jogadores[i]:<12} {agentes[i]:<10}  M1={probs_m1[i]:.1%}  M2={probs_m2[i]:.1%}")

    return {
        'prob_final':    prob_final,
        'prob_modelo1':  prob_m1,
        'prob_modelo2':  prob_m2,
        'prob_modelo3':  prob_m3,
        'por_jogador': {
            jogadores[i]: {'M1': probs_m1[i], 'M2': probs_m2[i]}
            for i in range(5)
        }
    }

In [61]:
stats_time = [
    {'R': 1.46, 'ACS': 237, 'K': 19, 'D': 11, 'A': 8,  '+/-': 8,  'KAST': 86, 'ADR': 174, 'HS%': 21, 'FK': 1, 'FD': 2, '+/-_FK_FD': -1},
    {'R': 1.30, 'ACS': 196, 'K': 16, 'D': 10, 'A': 10, '+/-': 6,  'KAST': 82, 'ADR': 127, 'HS%': 28, 'FK': 1, 'FD': 0, '+/-_FK_FD': 1},
    {'R': 1.03, 'ACS': 162, 'K': 12, 'D': 12, 'A': 5,  '+/-': 0,  'KAST': 82, 'ADR': 116, 'HS%': 35, 'FK': 2, 'FD': 1, '+/-_FK_FD': 1},
    {'R': 0.99, 'ACS': 233, 'K': 19, 'D': 17, 'A': 4,  '+/-': 2,  'KAST': 77, 'ADR': 142, 'HS%': 28, 'FK': 6, 'FD': 3, '+/-_FK_FD': 3},
    {'R': 0.88, 'ACS': 196, 'K': 14, 'D': 17, 'A': 14, '+/-': -3, 'KAST': 77, 'ADR': 138, 'HS%': 21, 'FK': 3, 'FD': 3, '+/-_FK_FD': 0},
]

resultado = ensemble_prever_time(
    mapa       = 'Ascent',
    jogadores  = ['BABYBAY', 'valyn', 'trent', 'jawgemo', 'leaf'],
    agentes    = ['Vyse', 'Astra', 'Fade', 'Neon', 'Phoenix'],
    roles      = ['sentinelas', 'controladores', 'iniciadores', 'duelistas', 'duelistas'],
    stats_time = stats_time,
    modelo1=modelo1, ohe=ohe, ohe_cols=ohe_cols, features1=features1, df_ml=df_ml,
    modelo2=modelo2, encoder=encoder, df_m2=df2, global_mean_m2=global_mean_modelo2,
    modelo3=modelo3, encoders=encoders, features3=features3,
    pesos=(0.92, 0.60, 0.90),
)

/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with f


══════════════════════════════════════════════════
  ENSEMBLE VALORANT — Ascent
══════════════════════════════════════════════════
  Modelo 1  stats individuais : 85.9%   peso=0.92
  Modelo 2  win rates         : 70.7%   peso=0.6
  Modelo 3  time agregado     : 59.3%   peso=0.9
──────────────────────────────────────────────────
  ► PROB. FINAL ENSEMBLE      : 72.2%
══════════════════════════════════════════════════

  Detalhes por jogador (M1 | M2):
    BABYBAY      Vyse        M1=95.0%  M2=58.9%
    valyn        Astra       M1=95.0%  M2=67.3%
    trent        Fade        M1=95.0%  M2=73.0%
    jawgemo      Neon        M1=79.0%  M2=77.2%
    leaf         Phoenix     M1=65.4%  M2=76.9%
